# Filtered aperture E/B shear maps on HEALPix (DP2, tract 9813)

Alex Broughton · April 2026. Rubin stack and Butler access required.


## Introduction

This notebook builds filtered aperture **E- and B-mode** shear maps on HEALPix for one tract of DP2 metadetect shear, using the Schirmer radial filter \(Q(r/R_s)\) with \(R_s\) from pixel scale settings in the config block below. The tree search radius is a multiple of \(R_s\) (it is not the same thing as \(R_s\)). Each map cell is a HEALPix RING pixel evaluated at its center; galaxies inside the search disc contribute with inverse-variance weights from the catalog. The **E-mode** map uses tangential shear \(g_t\) with \(Q\) (the usual aperture-mass statistic); the **B-mode** map uses the cross component \(g_\times\) with the same \(Q\). For a pure gravitational shear field, \(g_\times\) averages to zero in this construction. This is a weighted aperture statistic, not a Kaiser–Squires mass reconstruction.


## 1.0 Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

from lsst.daf.butler import Butler

CONFIG = {
    "REPO": "/sdf/data/rubin/repo/dp2_prep",  # Butler repository root
    "COLLECTION": "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",  # Input collection
    "SKYMAP": "lsst_cells_v2",  # Skymap name for Butler
    "INSTRUMENT": "LSSTCam",  # Instrument dimension
    "TRACT": 9813,  # Tract id
    "RS_INPUT_PIX": 10000,  # Schirmer scale in native catalog pixels
    "PIX_SCALE_ARCSEC": 0.2,  # Arcsec per pixel (converts RS_INPUT_PIX to sky angle)
    "SEARCH_RADIUS_RS_MULT": 3.0,  # cKDTree search radius = this × Rs (in arcmin)
    "NSIDE": 2048,  # HEALPix NSIDE (RING); use a power of 2
    "PEAK_SN_MIN": 3.0,  # Peak S/N cut; use None to disable
}

CONFIG["RS_ARCSEC"] = float(CONFIG["RS_INPUT_PIX"]) * float(CONFIG["PIX_SCALE_ARCSEC"])
CONFIG["RS_ARCMIN"] = CONFIG["RS_ARCSEC"] / 60.0
CONFIG["RS_DEG"] = CONFIG["RS_ARCSEC"] / 3600.0
CONFIG["THETA_MAX_ARCMIN"] = float(CONFIG["SEARCH_RADIUS_RS_MULT"]) * CONFIG["RS_ARCMIN"]
CONFIG["NSIDE"] = int(CONFIG["NSIDE"])
_mean_arcsec = float(np.rad2deg((4.0 * np.pi / 12.0) ** 0.5 / CONFIG["NSIDE"]) * 3600.0)

%matplotlib inline
plt.rcParams.update({"font.size": 12})

butler = Butler(
    CONFIG["REPO"],
    collections=[CONFIG["COLLECTION"]],
    skymap=CONFIG["SKYMAP"],
    instrument=CONFIG["INSTRUMENT"],
)

_npix = hp.nside2npix(CONFIG["NSIDE"])
print("TRACT =", CONFIG["TRACT"])
print(
    "Schirmer Rs =",
    CONFIG["RS_INPUT_PIX"],
    "pix @",
    CONFIG["PIX_SCALE_ARCSEC"],
    "arcsec/pix →",
    f"{CONFIG['RS_ARCMIN']:.3f} arcmin",
)
print(
    "search radius THETA_MAX_ARCMIN =",
    f"{CONFIG['THETA_MAX_ARCMIN']:.3f}",
    f"({CONFIG['SEARCH_RADIUS_RS_MULT']}×Rs) | NSIDE =",
    CONFIG["NSIDE"],
    f"(npix={_npix}; ~mean spacing {_mean_arcsec:.1f} arcsec)",
)


In [ ]:
def schirmer_weight(r, rs):
    """Schirmer filter Q(r/Rs); r and rs in the same angular units (degrees here)."""
    r = np.asarray(r, dtype=np.float64)
    rs = float(rs)
    x = r / rs
    a, b, c, d, xc = 6.0, 150.0, 47.0, 50.0, 0.15
    q = 1.0 / (1.0 + np.exp(a - b * x) + np.exp(d * x - c))
    ratio = x / xc
    with np.errstate(divide="ignore", invalid="ignore"):
        inner = np.tanh(ratio) / ratio
    inner = np.where(np.abs(ratio) < 1e-14, 1.0, inner)
    q = q * inner
    q = np.where(np.isfinite(q), q, 0.0)
    return q


def gamma_tx_sky(ra, dec, g1, g2, center):
    """Tangential g_t and cross g_x shear about center (degrees, small-angle plane)."""
    ra = np.asarray(ra, dtype=np.float64)
    dec = np.asarray(dec, dtype=np.float64)
    g1 = np.asarray(g1, dtype=np.float64)
    g2 = np.asarray(g2, dtype=np.float64)
    ra0, dec0 = center
    cos_dec0 = np.cos(np.deg2rad(dec0))
    dx = (ra - ra0) * cos_dec0
    dy = dec - dec0
    phi = np.arctan2(dy, dx)
    c2 = np.cos(2.0 * phi)
    s2 = np.sin(2.0 * phi)
    gt = -(g1 * c2 + g2 * s2)
    gx = -g1 * s2 + g2 * c2
    return gt, gx


def aperture_mass_and_sn_at_center(
    ra_c,
    dec_c,
    ra_all,
    dec_all,
    g1_all,
    g2_all,
    *,
    theta_max_arcmin,
    rs_deg,
    randomize=False,
    w_all=None,
):
    """Filtered aperture E- and B-mode statistics and S/N at one center; optional w_all weights galaxies.

    Returns (ap_e, sn_e, ap_b, sn_b) using Schirmer Q with tangential gt (E-type) and cross gx (B-type).
    """
    theta_max_deg = float(theta_max_arcmin) / 60.0
    dra = ra_all - ra_c
    ddec = dec_all - dec_c
    dtheta = np.hypot(dra, ddec)
    m = dtheta <= theta_max_deg
    if not np.any(m):
        return np.nan, np.nan, np.nan, np.nan

    ra_s = ra_all[m]
    dec_s = dec_all[m]
    g1_s = g1_all[m]
    g2_s = g2_all[m]
    d_theta = dtheta[m]
    n_g = int(ra_s.size)

    if w_all is None:
        w_s = np.ones(n_g, dtype=np.float64)
    else:
        w_s = np.asarray(w_all, dtype=np.float64)[m]

    gt, gx = gamma_tx_sky(ra_s, dec_s, g1_s, g2_s, center=(ra_c, dec_c))

    if randomize:
        rng = np.random.default_rng()
        ang = rng.uniform(0.0, 2.0 * np.pi)
        gt = gt * (-np.cos(ang))

    Qi = schirmer_weight(d_theta, rs_deg)
    w_sum = float(np.sum(w_s))
    if w_sum <= 0.0 or not np.isfinite(w_sum):
        return np.nan, np.nan, np.nan, np.nan

    ap_e = np.sum(Qi * gt * w_s) / w_sum
    ap_b = np.sum(Qi * gx * w_s) / w_sum
    denom = np.sqrt(np.sum((Qi**2) * (g1_s**2 + g2_s**2) * (w_s**2)))
    if denom <= 0.0 or not np.isfinite(denom):
        return ap_e, np.nan, ap_b, np.nan
    sn_e = np.sqrt(2.0) * (w_sum * ap_e) / denom
    sn_b = np.sqrt(2.0) * (w_sum * ap_b) / denom
    return ap_e, sn_e, ap_b, sn_b


def find_peaks_shear_healpix(
    ipix_eval,
    ap_mass,
    counts,
    sn_ratio,
    nside,
    theta_max_arcmin,
    sn_min=None,
):
    """Strict local maxima on HEALPix neighbors; density cut counts/(πθ²)>0.5 arcmin⁻²; optional sn_min."""
    ipix_eval = np.asarray(ipix_eval, dtype=np.int64)
    ap_mass = np.asarray(ap_mass, dtype=np.float64)
    counts = np.asarray(counts, dtype=np.float64)
    sn_ratio = np.asarray(sn_ratio, dtype=np.float64)
    idx_of = {int(ip): k for k, ip in enumerate(ipix_eval)}
    theta_max_arcmin = float(theta_max_arcmin)
    peaks_ipix = []

    for k, ip in enumerate(ipix_eval):
        ap = ap_mass[k]
        if not np.isfinite(ap):
            continue
        nbrs = hp.get_all_neighbours(nside, int(ip), nest=False)
        is_peak = True
        for nj in nbrs:
            if nj < 0:
                continue
            j = idx_of.get(int(nj))
            if j is None:
                continue
            if ap <= ap_mass[j]:
                is_peak = False
                break
        if not is_peak:
            continue
        density = counts[k] / (np.pi * (theta_max_arcmin**2))
        if density <= 0.5:
            continue
        if sn_min is not None:
            if not (np.isfinite(sn_ratio[k]) and sn_ratio[k] >= sn_min):
                continue
        peaks_ipix.append(int(ip))

    return np.asarray(peaks_ipix, dtype=np.int64)



## 2.0 Load data

Fetch the tract-level shear table from the Butler (one Astropy table for the tract).


In [ ]:
data = butler.get("object_shear_all", dataId={"tract": CONFIG["TRACT"]})
print(f"Rows: {len(data):,}")


## 3.0 Source selection



In [ ]:
mask = (data.columns['metaStep'] == "ns")
mask &= (data['image_flags'] == 0)
mask &= (data['psfOriginal_flags'] == 0)
mask &= (data['bmask_flags'] == 0)
mask &= (data['ormask_flags'] == 0)
mask &= (data['mfrac'] < 0.1)

# Mag cut?

shear_catalog = data[mask]
print("Sources before quality cuts:", len(data))
print("Sources after quality cuts:", len(shear_catalog))


## 4.0 Map on HEALPix

Unique HEALPix pixels that contain at least one selected galaxy are evaluated at pixel centers. For each center, gather galaxies within the search radius, apply the Schirmer weight to tangential shear \(g_t\) (E-type aperture statistic, `ap_mass_map` / `sn_ratio_map`) and to cross shear \(g_\times\) (B-type, `ap_bmode_map` / `sn_bmode_map`), and fill `ipix_eval`, `counts_map`, and those four map arrays.


In [ ]:
def _col(tbl, name):
    return np.array(tbl[name]).astype(np.float64, copy=False)

ra_deg = _col(shear_catalog, "ra")
dec_deg = _col(shear_catalog, "dec")
g1 = _col(shear_catalog, "gauss_g1")
g2 = _col(shear_catalog, "gauss_g2")

# Weights: 1 / (Var(g1) + Var(g2)) from catalog covariances.
_var_sum = _col(shear_catalog, "gauss_g1_g1_Cov") + _col(shear_catalog, "gauss_g2_g2_Cov")
with np.errstate(divide="ignore", invalid="ignore"):
    w_gal = np.where(np.isfinite(_var_sum) & (_var_sum > 0.0), 1.0 / _var_sum, 0.0)

print("N galaxies:", ra_deg.size)
print("Galaxies with positive finite weight:", int(np.sum(w_gal > 0)))


In [ ]:
theta_max_arcmin = CONFIG["THETA_MAX_ARCMIN"]
rs_deg = float(CONFIG["RS_DEG"])
nside = int(CONFIG["NSIDE"])
theta_max_deg = theta_max_arcmin / 60.0

ipix_gal = hp.ang2pix(nside, ra_deg, dec_deg, nest=False, lonlat=True)
ipix_eval = np.unique(ipix_gal)

xy = np.column_stack([ra_deg, dec_deg])
tree = cKDTree(xy)

n_cell = int(ipix_eval.size)
ap_mass_map = np.full(n_cell, np.nan, dtype=np.float64)
sn_ratio_map = np.full(n_cell, np.nan, dtype=np.float64)
ap_bmode_map = np.full(n_cell, np.nan, dtype=np.float64)
sn_bmode_map = np.full(n_cell, np.nan, dtype=np.float64)
counts_map = np.zeros(n_cell, dtype=np.int32)

for k, ip in enumerate(tqdm(ipix_eval, desc="HEALPix cells")):
    ra_c, dec_c = hp.pix2ang(nside, int(ip), nest=False, lonlat=True)
    idx = tree.query_ball_point([float(ra_c), float(dec_c)], r=theta_max_deg)
    if not idx:
        continue
    idx = np.asarray(idx, dtype=np.int64)
    ra_s = ra_deg[idx]
    dec_s = dec_deg[idx]
    g1_s = g1[idx]
    g2_s = g2[idx]
    counts_map[k] = int(ra_s.size)
    ap_mass_map[k], sn_ratio_map[k], ap_bmode_map[k], sn_bmode_map[k] = aperture_mass_and_sn_at_center(
        float(ra_c),
        float(dec_c),
        ra_s,
        dec_s,
        g1_s,
        g2_s,
        theta_max_arcmin=theta_max_arcmin,
        rs_deg=rs_deg,
        randomize=False,
        w_all=w_gal[idx],
    )

print("N HEALPix cells evaluated:", n_cell)
print("Finite E-mode (aperture mass) cells:", int(np.isfinite(ap_mass_map).sum()))
print("Finite B-mode cells:", int(np.isfinite(ap_bmode_map).sum()))


## 5.0 Peaks

Local maxima on the HEALPix neighbor graph, with the same surface-density cut as above and an optional S/N threshold from config.


In [ ]:
sn_min = CONFIG.get("PEAK_SN_MIN")
peak_ipix = find_peaks_shear_healpix(
    ipix_eval,
    ap_mass_map,
    counts_map,
    sn_ratio_map,
    nside,
    theta_max_arcmin=theta_max_arcmin,
    sn_min=sn_min,
)
print("N peaks (with current cuts):", int(peak_ipix.size))


## 6.0 Figures

RA/Dec scatter of aperture E-mode (mass) and B-mode statistics, S/N, and log counts for evaluated cells, plus S/N with peak positions. When the full HEALPix map is small enough in memory, gnomonic healpy views are also drawn.


In [ ]:
ra_pix, dec_pix = hp.pix2ang(nside, ipix_eval, nest=False, lonlat=True)
ra_pix = np.asarray(ra_pix, dtype=np.float64)
dec_pix = np.asarray(dec_pix, dtype=np.float64)

msk = counts_map > 0
if msk.any():
    m_lo, m_hi = np.nanpercentile(ap_mass_map[msk], [5, 95])
    sn_lo, sn_hi = np.nanpercentile(sn_ratio_map[msk], [5, 95])
    b_lo, b_hi = np.nanpercentile(ap_bmode_map[msk], [5, 95])
    snb_lo, snb_hi = np.nanpercentile(sn_bmode_map[msk], [5, 95])
else:
    m_lo = m_hi = sn_lo = sn_hi = b_lo = b_hi = snb_lo = snb_hi = None

pt_size = max(1.0, min(8.0, 500_000.0 / max(int(ipix_eval.size), 1)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
sc0 = axes[0].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=ap_mass_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=m_lo,
    vmax=m_hi,
    linewidths=0,
    alpha=0.85,
)
axes[0].set_xlabel("RA (deg)")
axes[0].set_ylabel("Dec (deg)")
axes[0].set_title(r"Aperture E-mode ($g_t \times Q$)")
axes[0].invert_xaxis()
fig.colorbar(sc0, ax=axes[0], fraction=0.046, pad=0.02)

sc1 = axes[1].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=sn_ratio_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=sn_lo,
    vmax=sn_hi,
    linewidths=0,
    alpha=0.85,
)
axes[1].set_xlabel("RA (deg)")
axes[1].set_ylabel("Dec (deg)")
axes[1].set_title(r"E-mode $\mathcal{S}/\mathcal{N}$")
axes[1].invert_xaxis()
fig.colorbar(sc1, ax=axes[1], fraction=0.046, pad=0.02)

logc = np.log10(counts_map.astype(np.float64) + 0.1)
sc2 = axes[2].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=logc[msk],
    s=pt_size,
    cmap="magma",
    linewidths=0,
    alpha=0.85,
)
axes[2].set_xlabel("RA (deg)")
axes[2].set_ylabel("Dec (deg)")
axes[2].set_title(r"$\log_{10}$(counts in search disc) + 0.1")
axes[2].invert_xaxis()
fig.colorbar(sc2, ax=axes[2], fraction=0.046, pad=0.02)

fig.suptitle(
    f"Tract {CONFIG['TRACT']}, NSIDE={nside}, Rs={CONFIG['RS_ARCMIN']:.2f}', search={theta_max_arcmin:.1f}'",
    y=1.02,
)

fig_eb, axes_eb = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)
scb0 = axes_eb[0].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=ap_bmode_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=b_lo,
    vmax=b_hi,
    linewidths=0,
    alpha=0.85,
)
axes_eb[0].set_xlabel("RA (deg)")
axes_eb[0].set_ylabel("Dec (deg)")
axes_eb[0].set_title(r"Aperture B-mode ($g_\times \times Q$)")
axes_eb[0].invert_xaxis()
fig_eb.colorbar(scb0, ax=axes_eb[0], fraction=0.046, pad=0.02)

scb1 = axes_eb[1].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=sn_bmode_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=snb_lo,
    vmax=snb_hi,
    linewidths=0,
    alpha=0.85,
)
axes_eb[1].set_xlabel("RA (deg)")
axes_eb[1].set_ylabel("Dec (deg)")
axes_eb[1].set_title(r"B-mode $\mathcal{S}/\mathcal{N}$")
axes_eb[1].invert_xaxis()
fig_eb.colorbar(scb1, ax=axes_eb[1], fraction=0.046, pad=0.02)
fig_eb.suptitle(
    f"Tract {CONFIG['TRACT']}, same filter/weights as E-mode row above",
    y=1.02,
)

fig2, axp = plt.subplots(figsize=(7, 5.5))
sc = axp.scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=sn_ratio_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=-4,
    vmax=4,
    linewidths=0,
    alpha=0.85,
)
if peak_ipix.size:
    ra_peak, dec_peak = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
    axp.scatter(
        ra_peak,
        dec_peak,
        s=120,
        facecolors="none",
        edgecolors="k",
        linewidths=1.2,
        label="peaks",
    )
axp.set_xlabel("RA (deg)")
axp.set_ylabel("Dec (deg)")
axp.set_title(r"E-mode $\mathcal{S}/\mathcal{N}$ with peak positions")
axp.invert_xaxis()
fig2.colorbar(sc, ax=axp, fraction=0.046, pad=0.02)
axp.legend(loc="upper right")
fig2.tight_layout()

# Optional gnomonic map if N_pix is modest.
MAX_DENSE_NPIX = 4_000_000
_npix_hp = hp.nside2npix(nside)
if _npix_hp <= MAX_DENSE_NPIX:
    rot_ra = float(np.median(ra_pix))
    rot_dec = float(np.median(dec_pix))
    sn_dense = np.full(_npix_hp, np.nan, dtype=np.float64)
    sn_dense[ipix_eval] = sn_ratio_map
    hp_sm = np.copy(sn_dense)
    hp_sm[~np.isfinite(hp_sm)] = hp.UNSEEN
    plt.figure(figsize=(7.0, 6.5))
    hp.gnomview(
        hp_sm,
        title=f"Tract {CONFIG['TRACT']} E-mode S/N (gnomonic)",
        rot=(rot_ra, rot_dec),
        xsize=600,
        reso=0.35,
        cmap="RdBu_r",
        min=-4,
        max=4,
        nest=False,
    )
    if peak_ipix.size:
        ra_pk, dec_pk = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
        hp.projscatter(
            ra_pk,
            dec_pk,
            lonlat=True,
            marker="o",
            s=80,
            facecolors="none",
            edgecolors="k",
            linewidths=1.0,
        )
    plt.show()

    snb_dense = np.full(_npix_hp, np.nan, dtype=np.float64)
    snb_dense[ipix_eval] = sn_bmode_map
    hp_smb = np.copy(snb_dense)
    hp_smb[~np.isfinite(hp_smb)] = hp.UNSEEN
    plt.figure(figsize=(7.0, 6.5))
    hp.gnomview(
        hp_smb,
        title=f"Tract {CONFIG['TRACT']} B-mode S/N (gnomonic)",
        rot=(rot_ra, rot_dec),
        xsize=600,
        reso=0.35,
        cmap="RdBu_r",
        min=-4,
        max=4,
        nest=False,
    )
    plt.show()
